In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from collections import defaultdict

class MultiTaskLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.head_short = nn.Linear(hidden_dim, 1)
        self.head_medium = nn.Linear(hidden_dim, 1)
        self.head_long = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        # x: (batch_size, seq_len, input_dim)
        lstm_out, _ = self.lstm(x)                     # (B, T, H)
        short  = self.head_short(lstm_out).squeeze(-1)  # (B, T)
        medium = self.head_medium(lstm_out).squeeze(-1) # (B, T)
        long   = self.head_long(lstm_out).squeeze(-1)   # (B, T)
        return short, medium, long
    
def create_sequences(df: pd.DataFrame, feature_cols, nan_cols, target_cols):
    sequences_X, sequences_Y = [], []
    for date in df['date_id'].unique():
        day_data = df[df['date_id'] == date].sort_values('minute_id')  # 确保顺序
        X = day_data[feature_cols + nan_cols].values.astype(np.float32)
        Y = day_data[target_cols].values.astype(np.float32)
        sequences_X.append(torch.tensor(X))
        sequences_Y.append(torch.tensor(Y))
    return sequences_X, sequences_Y

def get_batches(sequences_X, sequences_Y, length_to_indices, batch_size=32):
    batches = []
    for length, indices in length_to_indices.items():
        # 打乱顺序（可选，但推荐）
        np.random.shuffle(indices)
        # 分 batch
        for i in range(0, len(indices), batch_size):
            batch_indices = indices[i:i+batch_size]
            # stack 成 (B, T, D) 和 (B, T, 3)
            X_batch = torch.stack([sequences_X[idx] for idx in batch_indices])
            Y_batch = torch.stack([sequences_Y[idx] for idx in batch_indices])
            batches.append((X_batch, Y_batch))
    # 可选：打乱 batch 顺序
    np.random.shuffle(batches)
    return batches

In [2]:
df = pd.read_csv('dataset/processed_train.csv')
feature_cols = ['minute_id']+[f'feature_{i}' for i in range(1, 31)]
nan_cols = df.columns[df.columns.str.endswith('_nan')].tolist()
target_cols = ['target_short', 'target_medium', 'target_long']
sequences_X, sequences_Y = create_sequences(df, feature_cols, nan_cols, target_cols)

In [3]:
length_to_indices = defaultdict(list)
for i, seq in enumerate(sequences_X):
    length_to_indices[len(seq)].append(i)

In [4]:
# 初始化模型、优化器
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = len(feature_cols + nan_cols)
model = MultiTaskLSTM(input_dim=input_dim, hidden_dim=128).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# 获取所有 batch
batches = get_batches(sequences_X, sequences_Y, length_to_indices, batch_size=32)

# 训练
model.train()
for epoch in range(50):
    total_loss = 0
    for X_batch, Y_batch in batches:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)  # shape: (B, T, 3)
        
        optimizer.zero_grad()
        pred_s, pred_m, pred_l = model(X_batch)  # each: (B, T)
        
        # 提取真实值
        true_s = Y_batch[:, :, 0]
        true_m = Y_batch[:, :, 1]
        true_l = Y_batch[:, :, 2]
        
        # WMAE loss（假设 Y_batch 中无 NaN；如有需加 mask）
        loss = (0.5 * torch.mean(torch.abs(pred_s - true_s)) +
                0.3 * torch.mean(torch.abs(pred_m - true_m)) +
                0.2 * torch.mean(torch.abs(pred_l - true_l)))
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Avg Loss: {total_loss/len(batches):.6f}")

Epoch 1, Avg Loss: 0.030608
Epoch 2, Avg Loss: 0.015594
Epoch 3, Avg Loss: 0.013615
Epoch 4, Avg Loss: 0.013319
Epoch 5, Avg Loss: 0.011894
Epoch 6, Avg Loss: 0.011900
Epoch 7, Avg Loss: 0.011170
Epoch 8, Avg Loss: 0.011192
Epoch 9, Avg Loss: 0.010960
Epoch 10, Avg Loss: 0.010799
Epoch 11, Avg Loss: 0.010343
Epoch 12, Avg Loss: 0.010424
Epoch 13, Avg Loss: 0.010097
Epoch 14, Avg Loss: 0.010245
Epoch 15, Avg Loss: 0.009906
Epoch 16, Avg Loss: 0.010048
Epoch 17, Avg Loss: 0.009861
Epoch 18, Avg Loss: 0.010136
Epoch 19, Avg Loss: 0.009786
Epoch 20, Avg Loss: 0.009797
Epoch 21, Avg Loss: 0.009497
Epoch 22, Avg Loss: 0.009714
Epoch 23, Avg Loss: 0.009614
Epoch 24, Avg Loss: 0.009776
Epoch 25, Avg Loss: 0.009451
Epoch 26, Avg Loss: 0.009491
Epoch 27, Avg Loss: 0.009383
Epoch 28, Avg Loss: 0.009362
Epoch 29, Avg Loss: 0.009229
Epoch 30, Avg Loss: 0.009246
Epoch 31, Avg Loss: 0.009185
Epoch 32, Avg Loss: 0.009246
Epoch 33, Avg Loss: 0.009332
Epoch 34, Avg Loss: 0.009409
Epoch 35, Avg Loss: 0.0

In [8]:
df = pd.read_csv('dataset/processed_train.csv')
print("Median absolute targets:")
print(df[['target_short', 'target_medium', 'target_long']].abs().median())
print(df[['target_short', 'target_medium', 'target_long']].abs().median() @ np.array([0.5, 0.3, 0.2]))

Median absolute targets:
target_short     0.002509
target_medium    0.006922
target_long      0.017236
dtype: float64
0.0067780742248751205
